<a href="https://colab.research.google.com/github/grillinr/AI_Principles/blob/main/CS5173_HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS5173 HW2

# Step 1: Data

1. How many data samples are included in the
dataset?
2. Which problem will this dataset try to address?
3. What is the dimension ranging
in the dataset?
4. Does the dataset have any missing information? E.g., missing features.
5. What is the label of this dataset?
6. How many percent of data will you use for training,
validation and testing?
7. What kind of data pre-processing will you use for your training
dataset?

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

transform = transforms.Compose([transforms.Resize(256),
                                transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))])

train = datasets.ImageFolder("/content/drive/MyDrive/Data/train", transform=transform)
test = datasets.ImageFolder("/content/drive/MyDrive/Data/test", transform=transform)

cuda


In [2]:
# 1
print("1: ")
print(f"Train # of samples: {len(train)}")
print(f"Test # of samples: {len(test)}")

# 2
print("2: ")
print("This is a segmentation problem")

# 3
print("3: ")
print(f"Train dimension: {train[0][0].shape}")
print(f"Test dimension: {test[0][0].shape}")

# 4
print("4: ")
print("Train missing information: None")
print("Test missing information: None")

# 5
print("5: ")
print(f"Label: {train.classes}")

# 6
print("6: ")
print(f"Training Percent: {len(train) / (len(train) + len(test)) * 100:.0f}%")
print(f"Testing Percent: {len(test) / (len(train) + len(test)) * 100:.0f}%")

# 7
print("7: ")
print("Preprocessing: Resize to 256 and normalization")


1: 
Train # of samples: 160
Test # of samples: 40
2: 
This is a segmentation problem
3: 
Train dimension: torch.Size([3, 256, 256])
Test dimension: torch.Size([3, 256, 256])
4: 
Train missing information: None
Test missing information: None
5: 
Label: ['image', 'mask']
6: 
Training Percent: 80%
Testing Percent: 20%
7: 
Preprocessing: Resize to 256 and normalization


In [3]:
train_loader = torch.utils.data.DataLoader(train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=64, shuffle=True)

In [4]:
train_loader.dataset

Dataset ImageFolder
    Number of datapoints: 160
    Root location: /content/drive/MyDrive/Data/train
    StandardTransform
Transform: Compose(
               Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

# Step 2: Model

## U-Net Architecture
1. Conv layer with kernel 3
2. Conv layer with kernel 3
3. ReLU
4. Max pool

3 channels (R, G, B)
256x256 images

In [5]:
def double_conv2d(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(inplace=True)
    )

## U-Net 2-Block

In [6]:
class UNet2Block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet2Block, self).__init__()

        # Encoder
        self.enc1 = double_conv2d(in_channels, 64)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.enc2 = double_conv2d(64, 128)

        # Decoder
        self.upconv = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = double_conv2d(128, 64)  # 64 + 64 = 128 input
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)
        x2 = self.pool(x1)
        x3 = self.enc2(x2)

        # Decoder
        x4 = self.upconv(x3)
        x5 = torch.cat([x4, x1], dim=1)
        x6 = self.dec2(x5)
        out = self.out(x6)
        return out

## U-Net 3-Block

In [7]:
class UNet3Block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet3Block, self).__init__()

        # Encoder
        self.enc1 = double_conv2d(in_channels, 64)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.enc2 = double_conv2d(64, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.enc3 = double_conv2d(128, 256)

        # Decoder
        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = double_conv2d(256, 128)  # 128 + 128 = 256 input
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = double_conv2d(128, 64)   # 64 + 64 = 128 input
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)
        x2 = self.pool(x1)
        x3 = self.enc2(x2)
        x4 = self.pool2(x3)
        x5 = self.enc3(x4)

        # Decoder
        x6 = self.upconv3(x5)
        x7 = torch.cat([x6, x3], dim=1)
        x8 = self.dec3(x7)
        x9 = self.upconv2(x8)
        x10 = torch.cat([x9, x1], dim=1)
        x11 = self.dec2(x10)
        out = self.out(x11)
        return out

# Step 3: Objective

Loss function: Cross entropy

# Step 4: Optimization
Optimizer: Adam

# Step 5: Model Selection

1. U-Net 2 Blocks
2. U-Net 3 Blocks
3. Varying learning rates

In [8]:
in_channels = 3
out_channels = 2
lrs = [1e-2, 1e-3, 1e-4]
epochs = 100

# Step 6: Model demo

In [9]:
for model_class in [UNet2Block, UNet3Block]:
    for lr in lrs:
        model = model_class(in_channels, out_channels).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        print(f"\nTraining {model.__class__.__name__} with lr={lr}")
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for data, target in train_loader:
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            # Log every 10 epochs instead of every epoch
            if (epoch + 1) % 10 == 0:
                avg_loss = total_loss / len(train_loader)
                print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


Training UNet2Block with lr=0.01


RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of size: : [64]